# LeetCode 1209 — Remove All Adjacent Duplicates in String II bằng Python

Link đề: https://leetcode.com/problems/remove-all-adjacent-duplicates-in-string-ii/ · Độ khó: **Medium** · Chủ đề: String, Stack

Skeleton LeetCode:

```python
class Solution:
    def removeDuplicates(self, s: str, k: int) -> str:
        ...
```

Ghi chú: bản nâng cấp của LeetCode 1047 (xóa từng cặp trùng kề nhau). Ở đây xóa **khối `k` ký tự giống nhau liền kề**, và việc xóa có thể làm hai bên **dính lại** tạo khối mới. Lời giải chuẩn: **stack lưu cặp `[ký tự, số lần liên tiếp]`** — mỗi ký tự vào/ra stack tối đa một lần, `O(n)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lặp lại việc xóa `k` ký tự giống nhau liền kề cho đến khi không còn khối nào; kết quả là **duy nhất** (không phụ thuộc thứ tự xóa).
2. Thấy vì sao mô phỏng trực tiếp (quét – xóa – quét lại) tốn `O(n^2 / k)`.
3. Dùng **stack nén (run-length)**: mỗi phần tử stack là `[ký tự, đếm]`, tăng đếm khi gặp ký tự trùng đỉnh, pop khi đếm chạm `k`.
4. Biết cách làm **tại chỗ (in-place)** với con trỏ ghi `j` và mảng `count`, kỹ thuật "hai con trỏ đọc/ghi".
5. Tránh bẫy: chỉ quét một lượt mà không xử lý dây chuyền, so sánh với ký tự **gốc** thay vì đỉnh stack, quên reset đếm khi đổi ký tự.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`. Một phép **xóa k-trùng** là chọn `k` ký tự **liền kề và giống nhau** trong `s` rồi xóa chúng; phần bên trái và bên phải nối lại.

Lặp lại phép xóa cho đến khi không thực hiện được nữa. Trả về chuỗi cuối cùng. Đề đảm bảo đáp án là **duy nhất**.

**Ví dụ 1:** `s = "abcd"`, `k = 2` → `"abcd"` (không có gì để xóa).

**Ví dụ 2:** `s = "deeedbbcccbdaa"`, `k = 3` → `"aa"`.
- Xóa `"eee"` và `"ccc"` → `"ddbbbdaa"`; xóa `"bbb"` → `"dddaa"`; xóa `"ddd"` → `"aa"`.

**Ví dụ 3:** `s = "pbbcggttciiippooaais"`, `k = 2` → `"ps"`.

**Ràng buộc:** `1 <= s.length <= 10^5`; `2 <= k <= 10^4`; `s` chỉ gồm chữ thường.

In [1]:
import random
import time
from typing import List

examples = [
    ("abcd", 2, "abcd"),
    ("deeedbbcccbdaa", 3, "aa"),
    ("pbbcggttciiippooaais", 2, "ps"),
]
for s, k, want in examples:
    print(f"{s!r}, k={k} -> {want!r}")

'abcd', k=2 -> 'abcd'
'deeedbbcccbdaa', k=3 -> 'aa'
'pbbcggttciiippooaais', k=2 -> 'ps'


## 3. Kiến thức nền

### 3.1 Stack bằng list và truy cập đỉnh `stack[-1]`

`stack[-1]` là phần tử trên đỉnh (cuối list). `append` đẩy vào, `pop()` lấy ra — đều `O(1)`. Phần tử có thể là list có thể sửa (`stack[-1][1] += 1`).

In [2]:
stack = [['a', 1]]
stack.append(['b', 1])
stack[-1][1] += 1
print(stack, stack[-1])
stack.pop()
print(stack)

[['a', 1], ['b', 2]] ['b', 2]
[['a', 1]]


### 3.2 Nén chạy dài (run-length encoding)

Biểu diễn chuỗi bằng các "chạy" ký tự liên tiếp: `"aaabcc"` → `[('a',3), ('b',1), ('c',2)]`. `itertools.groupby` làm việc này sẵn.

In [3]:
from itertools import groupby
s = "deeedbbcccbdaa"
runs = [(ch, len(list(g))) for ch, g in groupby(s)]
print(runs)

[('d', 1), ('e', 3), ('d', 1), ('b', 2), ('c', 3), ('b', 1), ('d', 1), ('a', 2)]


### 3.3 Nhân chuỗi và ghép kết quả

`'a' * 3 == 'aaa'`. Dựng lại chuỗi từ stack nén: `''.join(ch * cnt for ch, cnt in stack)`.

In [4]:
stack = [['d', 2], ['a', 2]]
print(''.join(ch * cnt for ch, cnt in stack))

ddaa


### 3.4 Hai con trỏ đọc/ghi trên list (in-place)

Con trỏ `i` đọc lần lượt từng ký tự, con trỏ `j` là vị trí ghi tiếp theo. Ghi `arr[j] = arr[i]`, khi cần "xóa" chỉ việc lùi `j` — phần sau `j` coi như rác. Kết quả là `arr[:j]`.

In [5]:
arr = list("aabxc")
j = 0
for i in range(len(arr)):
    if arr[i] != 'x':          # ví dụ: bỏ mọi ký tự 'x'
        arr[j] = arr[i]
        j += 1
print(''.join(arr[:j]))

aabc


## 4. Ý tưởng nền tảng / trực giác

Tính tay `s = "deeedbbcccbdaa"`, `k = 3` theo kiểu stack nén (đọc từ trái sang phải):

| đọc | stack (ký tự, đếm) | ghi chú |
|---|---|---|
| d | (d,1) | |
| e e e | (d,1) (e,3) → pop | đủ 3 → xóa |
| d | (d,2) | `d` dính với `d` cũ! |
| b b | (d,2) (b,2) | |
| c c c | (d,2) (b,2) (c,3) → pop | |
| b | (d,2) (b,3) → pop | dây chuyền |
| d | (d,3) → pop | dây chuyền |
| a a | (a,2) | |

Kết quả `"aa"`. Nhận xét: ký tự mới chỉ cần so với **đỉnh stack** (phần đã "ổn định" bên trái), và khi đỉnh bị pop, ký tự tiếp theo tự nhiên so với phần tử bên dưới → dây chuyền được xử lý không cần quét lại.

In [6]:
s, k = "deeedbbcccbdaa", 3
stack = []
for ch in s:
    if stack and stack[-1][0] == ch:
        stack[-1][1] += 1
    else:
        stack.append([ch, 1])
    if stack[-1][1] == k:
        stack.pop()
    print(ch, [tuple(x) for x in stack])

d [('d', 1)]
e [('d', 1), ('e', 1)]
e [('d', 1), ('e', 2)]
e [('d', 1)]
d [('d', 2)]
b [('d', 2), ('b', 1)]
b [('d', 2), ('b', 2)]
c [('d', 2), ('b', 2), ('c', 1)]
c [('d', 2), ('b', 2), ('c', 2)]
c [('d', 2), ('b', 2)]
b [('d', 2)]
d []
a [('a', 1)]
a [('a', 2)]


## 5. Cách 1 — Brute Force / baseline

Mô phỏng đúng đề: quét chuỗi tìm khối `k` ký tự giống nhau liền kề đầu tiên, xóa, lặp lại đến khi không còn.

In [7]:
def brute(s: str, k: int) -> str:
    changed = True
    while changed:
        changed = False
        i = 0
        while i < len(s):
            j = i
            while j < len(s) and s[j] == s[i]:
                j += 1
            if j - i >= k:
                s = s[:i] + s[i + k:]      # xóa đúng k ký tự
                changed = True
                break
            i = j
    return s


for s, k, want in examples:
    print(brute(s, k), want)
    assert brute(s, k) == want

abcd abcd
aa aa
ps ps


**Độ phức tạp:** mỗi lần xóa tốn `O(n)` (quét + tạo chuỗi mới), có tới `n / k` lần xóa → `O(n^2 / k)`. Với `k = 2`, `n = 10^5`: `5·10^9` → **TLE**. Worst case: chuỗi dạng "lồng nhau" như `"abccba"` với `k = 2`, mỗi lần chỉ xóa được ở giữa.

**Nút thắt:** sau mỗi lần xóa ta **quét lại từ đầu**, trong khi chỉ vùng quanh chỗ nối mới có thể thay đổi.

In [8]:
def nested(m):
    letters = [chr(97 + i % 26) for i in range(m)]
    # tránh hai chữ cái giống nhau đứng cạnh khi chưa tới giữa: dùng xen kẽ
    left = [letters[i] if i % 2 == 0 else 'z' if letters[i] != 'z' else 'y' for i in range(m)]
    return ''.join(left) + ''.join(reversed(left))


for m in [500, 1000, 2000]:
    s = nested(m)
    t0 = time.perf_counter(); r = brute(s, 2); print(len(s), repr(r[:10]), f"{time.perf_counter() - t0:.3f}s")

1000 '' 0.022s
2000 '' 0.068s


4000 '' 0.270s


## 6. Bước đột phá

**Insight:** xử lý chuỗi từ trái sang phải; phần đã xử lý (bên trái) luôn ở trạng thái **ổn định** — không còn khối `k` nào. Khi đọc ký tự mới, chỉ có thể tạo khối mới **ở đuôi** phần ổn định. Vậy chỉ cần biết **ký tự cuối** và **độ dài chạy cuối** của phần ổn định → stack nén.

**Quy tắc:**
- **Nếu** ký tự mới bằng ký tự đỉnh **thì** tăng đếm của đỉnh; **ngược lại** đẩy `[ch, 1]`.
- **Nếu** đếm của đỉnh đạt `k` **thì** pop (xóa khối) — đỉnh mới lộ ra sẵn sàng nối với ký tự tiếp theo.

**Invariant:** stack là bản nén run-length của phần đã xử lý sau khi xóa hết, mọi đếm `< k`, và hai phần tử kề nhau trong stack có ký tự **khác nhau**.

**Vì sao thứ tự xóa không quan trọng?** Đề bảo đảm đáp án duy nhất; ta chỉ cần một thứ tự hợp lệ — stack thực hiện xóa "sớm nhất có thể từ trái qua".

## 7. Các cách tiếp theo

### Cách 2 — Stack các ký tự + stack đếm riêng (không nén)

Giữ `chars` (từng ký tự) và `counts[i]` = độ dài chạy kết thúc tại `chars[i]`. Khi `counts[-1] == k`, pop `k` lần. Dễ hiểu nhưng mỗi ký tự lưu riêng.

In [9]:
def solve_char_stack(s: str, k: int) -> str:
    chars, counts = [], []
    for ch in s:
        if chars and chars[-1] == ch:
            counts.append(counts[-1] + 1)
        else:
            counts.append(1)
        chars.append(ch)
        if counts[-1] == k:
            for _ in range(k):
                chars.pop(); counts.pop()
    return ''.join(chars)


for s, k, want in examples:
    assert solve_char_stack(s, k) == want
print([solve_char_stack(s, k) for s, k, _ in examples])

['abcd', 'aa', 'ps']


**Độ phức tạp:** mỗi ký tự push/pop một lần → `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 3 — Stack nén `[ký tự, đếm]` (lời giải chuẩn)

In [10]:
def solve_stack(s: str, k: int) -> str:
    stack = []
    for ch in s:
        if stack and stack[-1][0] == ch:
            stack[-1][1] += 1
            if stack[-1][1] == k:
                stack.pop()
        else:
            stack.append([ch, 1])
    return ''.join(ch * cnt for ch, cnt in stack)


for s, k, want in examples:
    assert solve_stack(s, k) == want
print([solve_stack(s, k) for s, k, _ in examples])

['abcd', 'aa', 'ps']


*(Chú ý: vì `k ≥ 2`, khi vừa đẩy `[ch, 1]` thì không thể đạt `k`, nên chỉ cần kiểm tra trong nhánh tăng đếm.)*

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (stack tối đa `n` phần tử khi không có gì trùng, ít hơn nhiều khi có chạy dài).

### Cách 4 — Hai con trỏ tại chỗ + mảng đếm

`arr = list(s)`, con trỏ ghi `j`, `count[j]` = độ dài chạy kết thúc tại `arr[j]`. Đọc `i`: ghi `arr[j] = arr[i]`, tính `count[j] = count[j-1] + 1` nếu `arr[j] == arr[j-1]` else `1`; nếu `count[j] == k` thì `j -= k` (xóa `k` ký tự vừa ghi), ngược lại `j += 1`.

In [11]:
def solve_two_pointers(s: str, k: int) -> str:
    arr = list(s)
    count = [0] * len(arr)
    j = 0
    for i in range(len(arr)):
        arr[j] = arr[i]
        count[j] = count[j - 1] + 1 if j > 0 and arr[j] == arr[j - 1] else 1
        if count[j] == k:
            j -= k - 1          # lùi về đầu khối: ô j - k + 1 sẽ bị ghi đè
            j -= 1
        j += 1
    return ''.join(arr[:j])


for s, k, want in examples:
    assert solve_two_pointers(s, k) == want
print([solve_two_pointers(s, k) for s, k, _ in examples])

['abcd', 'aa', 'ps']


Cách viết `j -= k - 1; j -= 1; j += 1` tương đương "nếu đủ `k` thì `j -= k - 1`, ngược lại `j += 1`" — lùi về vị trí đầu khối (sẽ bị ghi đè ở bước sau). Phiên bản gọn hơn ở mục 9 bài tập. **Độ phức tạp:** `O(n)` thời gian; bộ nhớ `O(n)` cho `arr` và `count` (trong ngôn ngữ có chuỗi khả biến như C++ có thể làm thẳng trên `s`).

### Cách 5 — Lặp theo chạy (run) cho tới ổn định

Cải tiến brute: mỗi lượt, nén `s` thành các chạy, giảm mỗi chạy về `cnt % k`, ghép lại; lặp đến khi chuỗi không đổi. Mỗi lượt `O(n)`; số lượt bằng **độ sâu dây chuyền** (có thể tới `O(n/k)` trong worst case) — minh họa vì sao cần stack.

In [12]:
def solve_rounds(s: str, k: int) -> str:
    rounds = 0
    while True:
        rounds += 1
        t = ''.join(ch * (len(list(g)) % k) for ch, g in groupby(s))
        if t == s:
            return s
        s = t


for s, k, want in examples:
    assert solve_rounds(s, k) == want
print([solve_rounds(s, k) for s, k, _ in examples])

['abcd', 'aa', 'ps']


**Vì sao `cnt % k` hợp lệ?** Một chạy dài `cnt` cho phép xóa `⌊cnt / k⌋` khối liên tiếp, còn lại `cnt % k`. Các xóa này đều hợp lệ trong đề; vì đáp án duy nhất, thực hiện chúng theo lượt vẫn cho cùng kết quả cuối.

## 8. TRACE từng bước

Trace Cách 3 trên `s = "pbbcggttciiippooaais"`, `k = 2`:

| ch | hành động | stack (nén) |
|---|---|---|
| p | đẩy | (p,1) |
| b,b | đẩy, tăng → 2 → pop | (p,1) |
| c | đẩy | (p,1)(c,1) |
| g,g | → pop | (p,1)(c,1) |
| t,t | → pop | (p,1)(c,1) |
| c | tăng (c,2) → pop | (p,1) |
| i,i | → pop | (p,1) |
| i | đẩy | (p,1)(i,1) |
| p,p | → pop | (p,1)(i,1) |
| o,o | → pop | (p,1)(i,1) |
| a,a | → pop | (p,1)(i,1) |
| i | tăng (i,2) → pop | (p,1) |
| s | đẩy | (p,1)(s,1) |

In [13]:
def trace_stack(s, k):
    stack = []
    for ch in s:
        if stack and stack[-1][0] == ch:
            stack[-1][1] += 1
            act = f"tăng -> {stack[-1][1]}"
            if stack[-1][1] == k:
                stack.pop(); act += " -> pop"
        else:
            stack.append([ch, 1]); act = "đẩy"
        print(f"{ch}: {act:16} stack = {' '.join(f'({c},{n})' for c, n in stack)}")
    return ''.join(c * n for c, n in stack)


print("=>", trace_stack("pbbcggttciiippooaais", 2))

p: đẩy              stack = (p,1)
b: đẩy              stack = (p,1) (b,1)
b: tăng -> 2 -> pop stack = (p,1)
c: đẩy              stack = (p,1) (c,1)
g: đẩy              stack = (p,1) (c,1) (g,1)
g: tăng -> 2 -> pop stack = (p,1) (c,1)
t: đẩy              stack = (p,1) (c,1) (t,1)
t: tăng -> 2 -> pop stack = (p,1) (c,1)
c: tăng -> 2 -> pop stack = (p,1)
i: đẩy              stack = (p,1) (i,1)
i: tăng -> 2 -> pop stack = (p,1)
i: đẩy              stack = (p,1) (i,1)
p: đẩy              stack = (p,1) (i,1) (p,1)
p: tăng -> 2 -> pop stack = (p,1) (i,1)
o: đẩy              stack = (p,1) (i,1) (o,1)
o: tăng -> 2 -> pop stack = (p,1) (i,1)
a: đẩy              stack = (p,1) (i,1) (a,1)
a: tăng -> 2 -> pop stack = (p,1) (i,1)
i: tăng -> 2 -> pop stack = (p,1)
s: đẩy              stack = (p,1) (s,1)
=> ps


In [14]:
def trace_two_pointers(s, k):
    arr = list(s); count = [0] * len(arr); j = 0
    for i in range(len(arr)):
        arr[j] = arr[i]
        count[j] = count[j - 1] + 1 if j > 0 and arr[j] == arr[j - 1] else 1
        full = count[j] == k
        j = j - k + 1 if full else j + 1
        print(f"i={i:2} '{s[i]}' -> {'xóa khối, ' if full else ''}j={j:2} hiện tại={''.join(arr[:j])}")
    return ''.join(arr[:j])


print("=>", trace_two_pointers("deeedbbcccbdaa", 3))

i= 0 'd' -> j= 1 hiện tại=d
i= 1 'e' -> j= 2 hiện tại=de
i= 2 'e' -> j= 3 hiện tại=dee
i= 3 'e' -> xóa khối, j= 1 hiện tại=d
i= 4 'd' -> j= 2 hiện tại=dd
i= 5 'b' -> j= 3 hiện tại=ddb
i= 6 'b' -> j= 4 hiện tại=ddbb
i= 7 'c' -> j= 5 hiện tại=ddbbc
i= 8 'c' -> j= 6 hiện tại=ddbbcc
i= 9 'c' -> xóa khối, j= 4 hiện tại=ddbb
i=10 'b' -> xóa khối, j= 2 hiện tại=dd
i=11 'd' -> xóa khối, j= 0 hiện tại=
i=12 'a' -> j= 1 hiện tại=a
i=13 'a' -> j= 2 hiện tại=aa
=> aa


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def removeDuplicates(self, s: str, k: int) -> str:
        stack = []                      # mỗi phần tử: [ký tự, số lần liên tiếp]
        for ch in s:
            if stack and stack[-1][0] == ch:
                stack[-1][1] += 1
                if stack[-1][1] == k:
                    stack.pop()
            else:
                stack.append([ch, 1])
        return ''.join(ch * cnt for ch, cnt in stack)


sol = Solution()
for s, k, want in examples:
    print(repr(sol.removeDuplicates(s, k)))
    assert sol.removeDuplicates(s, k) == want

'abcd'
'aa'
'ps'


## 10. Vì sao đúng

**Invariant:** sau khi đọc tiền tố `s[0..i]`, stack (giải nén) bằng kết quả cuối cùng của việc áp dụng quy trình xóa lên tiền tố đó; mọi đếm `< k`, các phần tử kề nhau khác ký tự.

- **Cơ sở:** stack rỗng = kết quả của chuỗi rỗng.
- **Bước:** gọi `R` là kết quả ổn định của tiền tố (không có khối `k` nào). Thêm `ch`: khối mới chỉ có thể xuất hiện ở đuôi, gồm chạy cuối của `R` (nếu cùng ký tự) và `ch`. Nếu chạy cuối đạt `k` thì xóa nó; phần còn lại vẫn là kết quả ổn định của một chuỗi không có khối `k` (vì hai phần tử kề nhau trong stack khác ký tự, việc pop không tạo khối mới ngay — khối mới chỉ có thể hình thành khi ký tự **tiếp theo** đến, và sẽ được xử lý ở bước sau).
- **Duy nhất:** đề bảo đảm mọi thứ tự xóa cho cùng đáp án, nên kết quả của thứ tự "từ trái qua" mà stack thực hiện chính là đáp án.

## 11. Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute (quét – xóa – quét lại) | `O(n^2 / k)` | `O(n)` |
| 2. Stack ký tự + stack đếm | `O(n)` | `O(n)` |
| 3. Stack nén | `O(n)` | `O(n)` (thường ít hơn) |
| 4. Hai con trỏ tại chỗ | `O(n)` | `O(n)` (trong Python) |
| 5. Lặp theo lượt | `O(n · số lượt)`, worst `O(n^2 / k)` | `O(n)` |

## 12. Bộ test

In [16]:
solvers = {
    "char_stack": solve_char_stack,
    "stack": solve_stack,
    "two_pointers": solve_two_pointers,
    "rounds": solve_rounds,
    "submit": Solution().removeDuplicates,
}
edge = [
    ("a", 2, "a"),
    ("aa", 2, ""),
    ("aaa", 2, "a"),
    ("aaaa", 2, ""),
    ("aaaaa", 3, "aa"),
    ("abba", 2, ""),
    ("abbac", 2, "c"),
    ("yfttttfbbbbnnnnffbgffffgbbbbgssssgthyyyy", 4, "ybth"),
    ("abcdabcd", 2, "abcdabcd"),
]
for s, k, want in examples + edge:
    assert brute(s, k) == want, (s, k, brute(s, k))
    for name, f in solvers.items():
        assert f(s, k) == want, (name, s, k, f(s, k), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [17]:
rng = random.Random(1209)
for trial in range(2000):
    k = rng.randint(2, 4)
    n = rng.randint(1, 18)
    s = ''.join(rng.choice('ab' if rng.random() < 0.5 else 'abc') for _ in range(n))
    want = brute(s, k)
    for name, f in solvers.items():
        assert f(s, k) == want, (name, s, k)
print("2000 test ngẫu nhiên khớp brute force")

2000 test ngẫu nhiên khớp brute force


In [18]:
rng = random.Random(4)
n = 10**5
cases = {
    "ngẫu nhiên 'ab', k=2": (''.join(rng.choice('ab') for _ in range(n)), 2),
    "lồng nhau, k=2": (nested(n // 2), 2),
    "toàn 'a', k=10^4": ('a' * n, 10**4),
    "26 chữ, k=3": (''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(n)), 3),
}
for label, (s, k) in cases.items():
    res = {}
    for name in ["char_stack", "stack", "two_pointers", "submit"]:
        t0 = time.perf_counter(); res[name] = solvers[name](s, k); dt = time.perf_counter() - t0
        print(f"{label:24s} {name:13s} len={len(res[name]):6} {dt:.3f}s")
    assert len(set(res.values())) == 1
assert Solution().removeDuplicates(nested(n // 2), 2) == ""
print("test giới hạn OK")

ngẫu nhiên 'ab', k=2     char_stack    len=   796 0.011s
ngẫu nhiên 'ab', k=2     stack         len=   796 0.008s
ngẫu nhiên 'ab', k=2     two_pointers  len=   796 0.012s
ngẫu nhiên 'ab', k=2     submit        len=   796 0.008s
lồng nhau, k=2           char_stack    len=     0 0.011s
lồng nhau, k=2           stack         len=     0 0.012s


lồng nhau, k=2           two_pointers  len=     0 0.012s
lồng nhau, k=2           submit        len=     0 0.010s


toàn 'a', k=10^4         char_stack    len=     0 0.011s
toàn 'a', k=10^4         stack         len=     0 0.008s


toàn 'a', k=10^4         two_pointers  len=     0 0.013s
toàn 'a', k=10^4         submit        len=     0 0.008s
26 chữ, k=3              char_stack    len= 99568 0.008s


26 chữ, k=3              stack         len= 99568 0.043s
26 chữ, k=3              two_pointers  len= 99568 0.012s
26 chữ, k=3              submit        len= 99568 0.017s
test giới hạn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ quét một lượt, không xử lý phản ứng dây chuyền

In [19]:
def wrong_one_pass(s, k):
    return ''.join(ch * (len(list(g)) % k) for ch, g in groupby(s))


print(repr(wrong_one_pass("deeedbbcccbdaa", 3)), "đúng là", repr(brute("deeedbbcccbdaa", 3)))

'ddbbbdaa' đúng là 'aa'


**Vì sao sai:** sau khi xóa `"eee"`, hai chữ `d` dính lại thành `"dd"` + ... tạo khối mới; một lượt không thấy. **Sửa:** stack (hoặc lặp đến ổn định như Cách 5).

### Lỗi 2 — Đếm theo ký tự **trước đó trong chuỗi gốc** thay vì đỉnh stack

In [20]:
def wrong_prev_original(s, k):
    stack = []
    cnt = 0
    for i, ch in enumerate(s):
        cnt = cnt + 1 if i > 0 and s[i - 1] == ch else 1     # SAI: so với s[i-1] gốc
        stack.append(ch)
        if cnt == k:
            del stack[-k:]
            cnt = 0
    return ''.join(stack)


print(repr(wrong_prev_original("deeedbbcccbdaa", 3)), "đúng là", repr(brute("deeedbbcccbdaa", 3)))

'ddbbbdaa' đúng là 'aa'


**Vì sao sai:** sau khi xóa, ký tự "đứng trước" ký tự hiện tại là **đỉnh stack**, không phải `s[i-1]`. Đếm theo chuỗi gốc bỏ lỡ những khối hình thành nhờ ghép nối. **Sửa:** lưu đếm cùng phần tử stack.

### Lỗi 3 — Không reset đếm khi đổi ký tự (dùng một biến đếm toàn cục)

In [21]:
def wrong_global_count(s, k):
    stack, cnt = [], 0
    for ch in s:
        if stack and stack[-1] == ch:
            cnt += 1
        else:
            cnt = 1
        stack.append(ch)
        if cnt == k:
            del stack[-k:]
            cnt = 0                              # SAI: đếm của chạy bên dưới bị mất
    return ''.join(stack)


print(repr(wrong_global_count("abbbaa", 3)), "đúng là", repr(brute("abbbaa", 3)))

'aaa' đúng là ''


**Vì sao sai:** sau khi xóa `"bbb"`, đỉnh là `a` với chạy dài `1`, nhưng biến `cnt` bị đặt `0`; hai chữ `a` tiếp theo đếm thành `1, 2` thay vì `2, 3` → không xóa được `"aaa"`. **Sửa:** mỗi chạy giữ đếm riêng trong stack (`[ch, cnt]`), pop thì đếm của chạy bên dưới vẫn còn nguyên.

### Lỗi 4 — Xóa toàn bộ chạy khi đạt `≥ k` thay vì đúng `k`

In [22]:
def wrong_remove_all(s, k):
    out = []
    for ch, g in groupby(s):
        n = len(list(g))
        if n < k:
            out.append(ch * n)                   # SAI: chạy dài >= k bị xóa sạch
    return ''.join(out)


print(repr(wrong_remove_all("aaaa", 3)), "đúng là", repr(brute("aaaa", 3)))

'' đúng là 'a'


**Vì sao sai:** mỗi phép xóa chỉ bỏ **đúng `k`** ký tự; chạy dài `4` với `k = 3` còn lại `1` ký tự. **Sửa:** giữ `cnt % k` (và vẫn phải xử lý dây chuyền).

## 14. Mẹo & nhận dạng pattern

- **"Xóa các phần tử kề nhau thỏa điều kiện, phần còn lại dính vào nhau, lặp lại"** → stack (phần đã xử lý ổn định nằm trong stack).
- **Cần đếm chạy liên tiếp** → stack nén `[giá trị, đếm]`.
- **Thao tác tại chỗ** → hai con trỏ đọc/ghi, lùi con trỏ ghi để "xóa".
- Cùng họ: 1047 (k = 2), 20 (ngoặc hợp lệ), 735 (va chạm thiên thạch), 2390 (xóa sao).

## 15. Biến thể

1. **LeetCode 1047 — Remove All Adjacent Duplicates In String:** `k = 2`.
2. **Xóa khi chạy đạt `≥ k` (xóa sạch cả chạy)** — kiểu "Candy Crush 1 chiều": chạy có thể dài thêm sau khi ghép, và kết quả **phụ thuộc thứ tự xóa** nên đề phải quy định rõ (xem Bài 5).
3. **Trên mảng số** thay vì chuỗi → y hệt.
4. **Đếm số phép xóa đã thực hiện** → cộng 1 mỗi lần pop.
5. **LeetCode 2696 — Minimum String Length After Removing Substrings** ("AB", "CD") → stack.

In [23]:
def remove_and_count(s, k):
    """Biến thể 4: trả về (chuỗi kết quả, số phép xóa)."""
    stack, removed = [], 0
    for ch in s:
        if stack and stack[-1][0] == ch:
            stack[-1][1] += 1
            if stack[-1][1] == k:
                stack.pop(); removed += 1
        else:
            stack.append([ch, 1])
    return ''.join(c * n for c, n in stack), removed


print(remove_and_count("deeedbbcccbdaa", 3), remove_and_count("aaaaaa", 2))
assert all(len(s) - len(remove_and_count(s, k)[0]) == k * remove_and_count(s, k)[1] for s, k, _ in examples + edge)

('aa', 4) ('', 3)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Tìm khối, xóa, quét lại | `O(n^2/k)` | `O(n)` | Oracle |
| 2. Stack ký tự + đếm | Pop `k` lần khi đủ | `O(n)` | `O(n)` | Dễ hiểu, gần 1047 |
| 3. Stack nén | `[ch, cnt]`, pop 1 lần | `O(n)` | `O(n)` | Nộp bài |
| 4. Hai con trỏ | Ghi đè, lùi `j` | `O(n)` | `O(n)` | Ngôn ngữ có chuỗi khả biến |
| 5. Lặp theo lượt | `cnt % k` mỗi lượt | `O(n·lượt)` | `O(n)` | Minh họa dây chuyền |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `removeDuplicates("abbbaaca", 3)` = ?

**Bài 2 (tính tay):** trace stack nén cho `s = "aabbbacd"`, `k = 3`.

**Bài 3 (điền TODO):** hoàn thiện `remove_adjacent_pairs(s)` (LeetCode 1047, `k = 2`) bằng stack ký tự đơn giản.

**Bài 4 (tự viết lại):** viết lại Cách 4 gọn hơn với một câu lệnh `j = j - k + 1 if count[j] == k else j + 1`... (thực chất là sửa `j` sau khi ghi).

**Bài 5 (mini challenge):** biến thể 2 — mỗi lần xóa **cả chạy** dài **≥ k** nằm **trái nhất**; lặp đến ổn định. Viết lời giải `O(n)` bằng stack. (Khác bài gốc, ở biến thể này **thứ tự xóa ảnh hưởng kết quả**: ví dụ `"babbbbabb"`, `k = 2` — xóa trái nhất trước cho `""`, còn xóa đồng loạt mọi chạy mỗi lượt cho `"b"`. Vì vậy phải cố định quy tắc. Gợi ý: chỉ quyết định xóa khi chạy **kết thúc**, tức gặp ký tự khác hoặc hết chuỗi.)

In [24]:
def remove_adjacent_pairs(s):
    # TODO: LeetCode 1047
    pass


def solve_two_pointers_compact(s, k):
    # TODO
    raise NotImplementedError


def crush_runs_at_least_k(s, k):
    # TODO: xóa cả chạy khi dài >= k, lặp đến ổn định, O(n)
    raise NotImplementedError

# assert remove_adjacent_pairs("abbaca") == "ca"
# assert crush_runs_at_least_k("abbbba", 3) == "aa"   # bbbb bị xóa, "aa" dài 2 < 3

## 18. Gợi ý 3 mức

- **Mức 1:** Ký tự mới chỉ cần so với phần đã ổn định bên trái — dùng stack.
- **Mức 2:** Lưu cả số lần liên tiếp trên đỉnh stack để biết khi nào đủ `k`.
- **Mức 3:** Bài 5: stack nén; khi gặp ký tự **khác** đỉnh, trước khi đẩy hãy kiểm tra đỉnh có `cnt ≥ k` không — nếu có thì pop, rồi so sánh lại ký tự mới với đỉnh mới (có thể gộp). Cuối chuỗi cũng kiểm tra đỉnh.

## 19. Đáp án tham khảo

**Bài 1:** `abbbaaca` → xóa `bbb` → `aaaca` → xóa `aaa` → `"ca"`.

**Bài 2:** a (a,1); a (a,2); b (a,2)(b,1); b (b,2); b (b,3)→pop → (a,2); a (a,3)→pop → []; c (c,1); d (c,1)(d,1) → `"cd"`.

In [25]:
assert Solution().removeDuplicates("abbbaaca", 3) == "ca"
assert Solution().removeDuplicates("aabbbacd", 3) == "cd"


def remove_adjacent_pairs(s):
    st = []
    for ch in s:
        if st and st[-1] == ch:
            st.pop()
        else:
            st.append(ch)
    return ''.join(st)


def solve_two_pointers_compact(s, k):
    arr, count, j = list(s), [0] * len(s), 0
    for i in range(len(s)):
        arr[j] = arr[i]
        count[j] = count[j - 1] + 1 if j and arr[j] == arr[j - 1] else 1
        j = j - k + 1 if count[j] == k else j + 1
    return ''.join(arr[:j])


def crush_runs_at_least_k(s, k):
    st = []
    for ch in s:
        if st and st[-1][0] == ch:
            st[-1][1] += 1
            continue
        if st and st[-1][1] >= k:            # chạy ở đỉnh đã kết thúc và đủ dài
            st.pop()
            if st and st[-1][0] == ch:
                st[-1][1] += 1
                continue
        st.append([ch, 1])
    if st and st[-1][1] >= k:
        st.pop()
    return ''.join(c * n for c, n in st)


def brute_crush(s, k):
    """Mô phỏng: mỗi lần xóa chạy dài >= k ĐẦU TIÊN từ trái, lặp đến ổn định."""
    while True:
        pos = 0
        for ch, g in groupby(s):
            n = len(list(g))
            if n >= k:
                s = s[:pos] + s[pos + n:]
                break
            pos += n
        else:
            return s


assert remove_adjacent_pairs("abbaca") == "ca"
assert crush_runs_at_least_k("abbbba", 3) == "aa"
rng = random.Random(19)
for _ in range(2000):
    s = ''.join(rng.choice('abc') for _ in range(rng.randint(0, 14)))
    k = rng.randint(2, 4)
    assert remove_adjacent_pairs(s) == brute(s, 2)
    assert solve_two_pointers_compact(s, k) == brute(s, k)
    assert crush_runs_at_least_k(s, k) == brute_crush(s, k), (s, k)
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
stack = []                       # [ký tự, đếm]
for ch in s:
    if stack and stack[-1][0] == ch:
        stack[-1][1] += 1
        if stack[-1][1] == k: stack.pop()
    else:
        stack.append([ch, 1])
return ''.join(c * n for c, n in stack)
```

**Checklist:**
- [ ] So sánh với **đỉnh stack**, không phải ký tự trước trong chuỗi gốc.
- [ ] Mỗi chạy giữ **đếm riêng** (pop xong đếm của chạy dưới còn nguyên).
- [ ] Xóa **đúng `k`** ký tự, phần dư giữ lại.
- [ ] Dây chuyền (xóa xong hai bên dính lại) được xử lý tự động.
- [ ] Kết quả có thể là chuỗi rỗng `""`.
- [ ] Ghép kết quả bằng `''.join`.